# VSD Ground-Truth STL Generator (TotalSegmentator)

Automates the manual 3D-Slicer per-bone segmentation for the VSD dataset. For each raw VSD knee
CT this notebook:

1. Runs **TotalSegmentator** twice — `total` (roi_subset femur) for the **femur**, and
   `appendicular_bones` for **patella / tibia / fibula**.
2. Applies **3.00 mm Gaussian** smoothing to each bone label map.
3. Runs three **quality gates** (flag → log only, skip export, leave for manual Slicer work):
   - **extra_labels** — > 4 distinct appendicular labels that are *not* patella/tibia/fibula.
   - **bilateral_femur** — both `femur_left` and `femur_right` present in one single-leg case.
   - **fibula_disconnected** — smoothed fibula splits into > 1 connected component.
4. Shows a **QA visual** (coronal + sagittal MIP with the 4 bones overlaid) for **every** case.
5. Writes per-bone **STL** meshes (raw-CT LPS world frame) following the `VSD_002` convention:
   `data/external/ground_truth/VSD_ground_truth/<CaseID>/<Side>/VSD_<id>_<Side> segmentation_<bone>.stl`.

**Frame:** meshes are the exact inverse of `notebooks/modeling/gt_per_bone.ipynb::voxelize_on`
(marching-cubes index `(z,y,x)` → reverse → `TransformContinuousIndexToPhysicalPoint` → world LPS),
so the downstream voxelizer consumes them unchanged.

**Environment:** written for **HPC GPU** (`DEVICE="gpu"`, `FAST=False`). Set `DEVICE="cpu"`,
`FAST=True`, `CASE_LIMIT=1` in the CONFIG cell for a local smoke test. POSIX paths.

> **License:** `appendicular_bones` needs a free non-commercial TotalSegmentator license.
> Run once in the venv: `totalseg_set_license -l <YOUR_KEY>` (or set env `TOTALSEG_LICENSE`).

In [ ]:
# ============================================================
# CONFIG  — the only cell to edit when switching local <-> HPC
# ============================================================
from pathlib import Path

# --- environment ---
DEVICE = "gpu"      # "gpu" on HPC | "cpu" for a local smoke test
FAST   = False      # False = full model (HPC/GPU). Set True for a tractable local CPU smoke test.
CASE_LIMIT = None   # None = all cases | e.g. 1 to smoke-test a single case

# --- project root (POSIX; digital-twin maps HPC notebooks/ <-> HPC_notebooks/pre_processing_HPC/) ---
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), f"could not locate project data/ from {Path.cwd()}"

# --- inputs ---
RAW_HEALTHY = ROOT / "data/raw/healthy"                       # VSD.<id>/VSD_<id>_<Side>.nii.gz (raw HU)
Z036_EXTERNAL = ROOT / "data/external/manual_contralateral_healthy/VSD_z036_Left_manual.nrrd"

# --- outputs ---
GT_ROOT  = ROOT / "data/external/ground_truth/VSD_ground_truth"       # STL output (VSD_002 convention)
FIG_DIR  = ROOT / "reports/figures/vsd_ts_ground_truth"              # QA PNGs
REPORT_DIR = GT_ROOT / "_reports"                                    # manifest.csv / flags.csv / diagnostics.csv
SCRATCH  = ROOT / "data/interim/_ts_scratch"                         # temp TS output + nrrd->nii
MASK_DIR = ROOT / "data/interim/ts_bone_masks"                       # persisted per-bone masks (feeds fallback)
SAVE_MASKS = "all"                                                  # "all" | "flagged": which cases' masks to persist
for d in (FIG_DIR, REPORT_DIR, SCRATCH, MASK_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- bone / label constants ---
BONES = ["femur", "tibia", "patella", "fibula"]                      # canonical order
FEMUR_LABELS = ["femur_left", "femur_right"]                         # TS `total` task
APPENDICULAR_TARGETS = {"patella": "patella", "tibia": "tibia", "fibula": "fibula"}  # TS `appendicular_bones`
BONE_COLORS = {"femur": (1.0, 0.30, 0.30), "tibia": (0.30, 0.70, 1.0),
               "patella": (1.0, 0.85, 0.20), "fibula": (0.45, 1.0, 0.45)}

# --- smoothing / meshing ---
SMOOTH_MM = 3.00           # Gaussian std-dev (mm) on each label map before marching cubes (Slicer semantics)
PAD = 2                    # zero-pad voxels before marching cubes -> caps FOV-cut ends -> watertight solids
MIN_VOXELS = 100           # a label / component smaller than this is treated as absent / noise

# --- diagnostics thresholds (A3) ---
EXTRA_LABEL_LIMIT = 4      # > this many non-target appendicular labels -> extra_labels
HU_BONE = 150             # HU above which a voxel counts as bone (landing score)
HU_MIN_FRAC = 0.50        # < this fraction of a mask on bone-HU -> low_hu (soft)
OVERLAP_FRAC = 0.10       # (bone_i & bone_j) / min(|bone_i|,|bone_j|) above this -> overlap (hard)
EXPECT_INTERIOR = {"patella", "fibula"}   # these should NOT touch a volume face; femur/tibia are FOV-cut
# plausible per-bone physical volume (mm^3). Wide bounds; femur/tibia are cut by the crop FOV. Tune from cohort.
VOLUME_RANGE_MM3 = {"femur": (3e4, 6e5), "tibia": (3e4, 6e5),
                    "patella": (8e2, 6e4), "fibula": (8e2, 9e4)}

# --- flag severity: HARD -> skip export (route to fallback / manual) ; SOFT -> warn, still export ---
HARD_FLAGS = {"bilateral_femur", "extra_labels", "missing", "disconnected", "truncated", "overlap"}
SOFT_FLAGS = {"low_hu", "volume_outlier", "side_mismatch"}

# --- excluded cases ---
EXCLUDE = {"VSD_002_Left", "VSD_002_Right",   # manual reference already done
           "VSD_z050_Right", "VSD_z063_Right"}  # TKR exclusions

print(f"ROOT        = {ROOT}")
print(f"DEVICE={DEVICE}  FAST={FAST}  CASE_LIMIT={CASE_LIMIT}  SAVE_MASKS={SAVE_MASKS}")
print(f"GT_ROOT     = {GT_ROOT}")
print(f"SMOOTH_MM={SMOOTH_MM}  PAD={PAD}  MIN_VOXELS={MIN_VOXELS}  EXTRA_LABEL_LIMIT={EXTRA_LABEL_LIMIT}")

In [ ]:
import shutil, tempfile
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage import measure
import trimesh
import matplotlib.pyplot as plt
from totalsegmentator.python_api import totalsegmentator

print("imports OK — TotalSegmentator, trimesh, skimage.measure loaded")

## 1. Case discovery

Enumerate every raw single-leg VSD volume (`VSD.<id>/VSD_<id>_<Side>.nii.gz`), drop the excluded
cases, and substitute the external manual volume for `VSD_z036_Left`. Each work item is
`(volume_id, case_id, side, input_path)` where `case_id` = `VSD_<id>` (the output folder name).

In [ ]:
def discover_cases():
    """Return sorted work list of dicts: volume_id, case_id, side, input_path."""
    work = []
    for vol in sorted(RAW_HEALTHY.glob("VSD.*/VSD_*_*.nii.gz")):
        stem = vol.name.replace(".nii.gz", "")          # e.g. VSD_z036_Left
        if stem in EXCLUDE:
            continue
        case_id, side = stem.rsplit("_", 1)             # ("VSD_z036", "Left")
        assert side in ("Left", "Right"), f"unexpected side in {vol.name}"
        src = vol
        if stem == "VSD_z036_Left":                     # use the external manual volume, not raw/healthy
            assert Z036_EXTERNAL.exists(), f"missing {Z036_EXTERNAL}"
            src = Z036_EXTERNAL
        work.append(dict(volume_id=stem, case_id=case_id, side=side, input_path=src))

    # de-dup on volume_id (z036 substitution can otherwise collide) keeping the substituted path
    seen = {}
    for w in work:
        seen[w["volume_id"]] = w
    work = sorted(seen.values(), key=lambda w: w["volume_id"])
    if CASE_LIMIT is not None:
        work = work[:CASE_LIMIT]
    return work

CASES = discover_cases()
print(f"{len(CASES)} cases to process")
for w in CASES:
    tag = "  (external nrrd)" if w["input_path"].suffix == ".nrrd" else ""
    print(f"  {w['volume_id']:18s} <- {w['input_path'].relative_to(ROOT)}{tag}")

## 2. Helpers

Segmentation, smoothing, connectivity, and the marching-cubes → world-LPS STL exporter (the exact
inverse of `gt_per_bone.ipynb::voxelize_on`). All masks carry the input CT geometry, so a single
`ct_img` is the geometric reference for smoothing sigma and vertex → world mapping.

In [ ]:
def ensure_nifti(path, workdir):
    """TS reads NIfTI; convert .nrrd (VSD_z036) to a temp .nii.gz preserving HU + geometry."""
    if path.name.endswith(".nii") or path.name.endswith(".nii.gz"):
        return path
    img = sitk.ReadImage(str(path))
    out = workdir / "input.nii.gz"
    sitk.WriteImage(img, str(out))
    return out


def run_ts(nifti_path, out_dir, task, roi_subset=None):
    """Run TotalSegmentator (separate binary masks per class) into out_dir."""
    out_dir.mkdir(parents=True, exist_ok=True)
    totalsegmentator(
        input=str(nifti_path), output=str(out_dir), task=task,
        roi_subset=roi_subset, fast=FAST, device=DEVICE, ml=False, quiet=True,
    )
    return out_dir


def load_label(out_dir, name):
    """Return binary array (z,y,x) uint8 for a TS output label, or None if absent/empty-file."""
    p = out_dir / f"{name}.nii.gz"
    if not p.exists():
        return None
    return (sitk.GetArrayFromImage(sitk.ReadImage(str(p))) > 0).astype(np.uint8)


def present_labels(out_dir, min_voxels=MIN_VOXELS):
    """{label_name: voxel_count} for every TS output mask with count >= min_voxels."""
    found = {}
    for p in sorted(out_dir.glob("*.nii.gz")):
        n = int((sitk.GetArrayFromImage(sitk.ReadImage(str(p))) > 0).sum())
        if n >= min_voxels:
            found[p.name[:-len(".nii.gz")]] = n
    return found


def smooth_field(arr, ct_img):
    """Gaussian-smooth a binary label map with SMOOTH_MM std-dev (mm). Returns float field in [0,1]."""
    sx, sy, sz = ct_img.GetSpacing()                       # sitk (x,y,z)
    sigma = (SMOOTH_MM / sz, SMOOTH_MM / sy, SMOOTH_MM / sx)  # array (z,y,x)
    return ndimage.gaussian_filter(arr.astype(np.float32), sigma=sigma)


def n_components(binary):
    """Number of 26-connected components with >= MIN_VOXELS voxels."""
    lab, n = ndimage.label(binary, structure=ndimage.generate_binary_structure(3, 3))
    if n == 0:
        return 0
    sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))
    return int((sizes >= MIN_VOXELS).sum())


def field_to_mesh(field, ct_img, level=0.5, pad=PAD):
    """Marching cubes on a (smoothed) field (z,y,x) -> WATERTIGHT trimesh in world-LPS mm.

    Zero-padding by `pad` forces a closed isosurface, so FOV-cut shaft ends get a flat cap
    (watertight solid). Inverse of voxelize_on: verts (z,y,x) padded index -> minus pad ->
    reverse to sitk (i,j,k)=(x,y,z) -> TransformContinuousIndexToPhysicalPoint -> world LPS.
    """
    if float(field.max()) < level:
        return None
    padded = np.pad(field, pad, mode="constant", constant_values=0.0)
    verts, faces, _, _ = measure.marching_cubes(padded, level=level)
    verts = verts - pad                                    # padded index -> original array index
    idx = verts[:, ::-1]                                   # (z,y,x) -> (x,y,z) sitk index order
    world = np.array([ct_img.TransformContinuousIndexToPhysicalPoint(tuple(map(float, p)))
                      for p in idx], dtype=np.float64)
    return trimesh.Trimesh(vertices=world, faces=faces, process=False)


# ---------- diagnostics helpers (A3) ----------

def touched_faces(binary):
    """Set of volume faces the mask touches: {'z0','z1','y0','y1','x0','x1'}."""
    t = set()
    if binary.sum() == 0:
        return t
    if binary[0].any():  t.add("z0")
    if binary[-1].any(): t.add("z1")
    if binary[:, 0].any():  t.add("y0")
    if binary[:, -1].any(): t.add("y1")
    if binary[:, :, 0].any():  t.add("x0")
    if binary[:, :, -1].any(): t.add("x1")
    return t


def hu_score(binary, ct_arr, hu_bone=HU_BONE):
    """Fraction of mask voxels that sit on bone-bright CT (HU > hu_bone)."""
    n = int(binary.sum())
    return float((ct_arr[binary > 0] > hu_bone).mean()) if n else 0.0


def volume_mm3(binary, ct_img):
    """Physical volume of a binary mask in mm^3."""
    sx, sy, sz = ct_img.GetSpacing()
    return float(int(binary.sum()) * sx * sy * sz)


def pairwise_overlap_frac(a, b):
    """|a & b| / min(|a|,|b|); 0 if either is empty."""
    na, nb = int(a.sum()), int(b.sum())
    if na == 0 or nb == 0:
        return 0.0
    return float(int((a & b).sum()) / min(na, nb))


def save_mask(binary, ct_img, path):
    """Write a binary mask (z,y,x) as compressed uint8 .nii.gz carrying the CT geometry."""
    path.parent.mkdir(parents=True, exist_ok=True)
    im = sitk.GetImageFromArray(binary.astype(np.uint8))
    im.CopyInformation(ct_img)
    sitk.WriteImage(im, str(path), useCompression=True)


print("helpers loaded: ensure_nifti, run_ts, load_label, present_labels, smooth_field, n_components, "
      "field_to_mesh(+cap), touched_faces, hu_score, volume_mm3, pairwise_overlap_frac, save_mask")

In [ ]:
def _mip(vol, axis):
    """2D maximum-intensity projection, transposed for lower-origin display (matches repo idiom)."""
    return np.max(vol, axis=axis).T


def qa_visual(volume_id, ct_arr, masks, status_str, save_path):
    """Coronal (AP) + sagittal (LAT) MIP of the CT with the 4 bone masks overlaid in distinct colors."""
    fig, axes = plt.subplots(1, 2, figsize=(9, 5))
    for ax, axis, name in ((axes[0], 1, "AP"), (axes[1], 2, "LAT")):
        ax.imshow(_mip(ct_arr, axis), cmap="gray", vmin=-450, vmax=1050, origin="lower")
        for bone in BONES:
            m = masks.get(bone)
            if m is None or m.sum() < MIN_VOXELS:
                continue
            mip = _mip(m, axis) > 0
            rgba = np.zeros((*mip.shape, 4), dtype=np.float32)
            rgba[mip, :3] = BONE_COLORS[bone]
            rgba[mip, 3] = 0.50
            ax.imshow(rgba, origin="lower")
        ax.set_title(f"{name}", fontsize=10)
        ax.axis("off")
    present = ", ".join(b for b in BONES if masks.get(b) is not None and masks[b].sum() >= MIN_VOXELS)
    color = "green" if status_str.startswith("PASS") and "WARN" not in status_str else \
            ("darkorange" if status_str.startswith("PASS_WARN") else "red")
    fig.suptitle(f"{volume_id}   [{status_str}]\nbones: {present or 'none'}", fontsize=11, color=color)
    handles = [plt.Line2D([0], [0], marker="s", ls="", markersize=9, color=BONE_COLORS[b], label=b)
               for b in BONES]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(rect=(0, 0.05, 1, 0.92))
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


print("qa_visual loaded")

## 3. Per-case pipeline

Per case: segment (`total` femur + `appendicular_bones`) → select femur (**bilateral_femur** /
**side_mismatch**) → **extra_labels** → smooth all 4 bones (3 mm) → **per-bone scored diagnostics**
(components, boundary/**truncated**, **hu_score**/`low_hu`, **volume**/`volume_outlier`) →
inter-bone **overlap** → persist raw masks → QA visual (always) → **export capped watertight STLs**
unless HARD-flagged.

Status: **PASS** (clean) / **PASS_WARN** (soft issues only, still exported) / **FLAG** (≥1 HARD
issue → skip export, route to `vsd_gt_watertight.ipynb` or manual). Severity is set by
`HARD_FLAGS`/`SOFT_FLAGS` in CONFIG. Every case gets a QA PNG and a numeric row in `diagnostics.csv`.

In [ ]:
def process_case(w):
    """Segment, persist masks, score diagnostics, QA, and (if not HARD-flagged) export 4 capped
    watertight STLs. Returns (manifest_row, [per-bone diagnostic rows])."""
    vid, case_id, side, in_path = w["volume_id"], w["case_id"], w["side"], w["input_path"]
    workdir = Path(tempfile.mkdtemp(prefix=f"{vid}_", dir=SCRATCH))
    row = dict(volume_id=vid, case_id=case_id, side=side, status="", hard_flags="", soft_flags="",
               n_extra_labels=0, extra_labels="", femur_left=0, femur_right=0)
    diag_rows = []
    try:
        nifti = ensure_nifti(in_path, workdir)
        ct_img = sitk.ReadImage(str(nifti))
        ct_arr = sitk.GetArrayFromImage(ct_img)               # HU (z,y,x)

        total_dir = run_ts(nifti, workdir / "total", task="total",
                           roi_subset=(FEMUR_LABELS if not FAST else None))
        app_dir = run_ts(nifti, workdir / "appendicular", task="appendicular_bones")

        hard, soft = [], []

        # --- femur: select present side; bilateral + side-mismatch checks ---
        fl, fr = load_label(total_dir, "femur_left"), load_label(total_dir, "femur_right")
        nfl = int(fl.sum()) if fl is not None else 0
        nfr = int(fr.sum()) if fr is not None else 0
        row["femur_left"], row["femur_right"] = nfl, nfr
        if nfl >= MIN_VOXELS and nfr >= MIN_VOXELS:
            hard.append("bilateral_femur")
        femur = (fl if nfl >= nfr else fr)
        anat_side = "left" if nfl >= nfr else "right"
        if femur is None or max(nfl, nfr) < MIN_VOXELS:
            femur = np.zeros(ct_arr.shape, np.uint8)
        elif anat_side != side.lower():                       # single-leg crop: TS side should match the leg
            soft.append("side_mismatch")

        # --- appendicular targets + extra-label check ---
        appen = present_labels(app_dir)
        extra = sorted(k for k in appen if k not in APPENDICULAR_TARGETS)
        row["n_extra_labels"], row["extra_labels"] = len(extra), ";".join(extra)
        if len(extra) > EXTRA_LABEL_LIMIT:
            hard.append("extra_labels")

        raw = {"femur": femur}
        for bone, ts_name in APPENDICULAR_TARGETS.items():
            m = load_label(app_dir, ts_name)
            raw[bone] = m if m is not None else np.zeros(ct_arr.shape, np.uint8)

        # --- 3 mm Gaussian smoothing on all bones ---
        fields = {b: smooth_field(raw[b], ct_img) for b in BONES}
        binar = {b: (fields[b] >= 0.5).astype(np.uint8) for b in BONES}

        # --- per-bone scored diagnostics ---
        for b in BONES:
            m = binar[b]
            present = int(m.sum()) >= MIN_VOXELS
            comps = n_components(m)
            faces = touched_faces(m)
            hu = hu_score(m, ct_arr)
            vol = volume_mm3(m, ct_img)
            vlo, vhi = VOLUME_RANGE_MM3[b]
            diag_rows.append(dict(volume_id=vid, bone=b, present=int(present), voxels=int(m.sum()),
                                  components=comps, faces="|".join(sorted(faces)),
                                  hu_score=round(hu, 3), volume_mm3=round(vol, 1)))
            row[f"{b}_voxels"] = int(m.sum())
            row[f"{b}_components"] = comps
            row[f"{b}_hu"] = round(hu, 3)
            if not present:
                hard.append(f"missing:{b}")
                continue
            if comps > 1:
                hard.append(f"disconnected:{b}")
            if b in EXPECT_INTERIOR and faces:                # patella/fibula cut off by FOV -> incomplete
                hard.append(f"truncated:{b}")
            if hu < HU_MIN_FRAC:
                soft.append(f"low_hu:{b}")
            if not (vlo <= vol <= vhi):
                soft.append(f"volume_outlier:{b}")

        # --- inter-bone overlap (mislabeled boundaries) ---
        for i in range(len(BONES)):
            for j in range(i + 1, len(BONES)):
                bi, bj = BONES[i], BONES[j]
                if pairwise_overlap_frac(binar[bi], binar[bj]) > OVERLAP_FRAC:
                    hard.append(f"overlap:{bi}-{bj}")

        hard, soft = sorted(set(hard)), sorted(set(soft))
        status = "FLAG" if hard else ("PASS_WARN" if soft else "PASS")
        row["status"], row["hard_flags"], row["soft_flags"] = status, ";".join(hard), ";".join(soft)

        # --- persist raw per-bone masks (feeds vsd_gt_watertight.ipynb) ---
        if SAVE_MASKS == "all" or (SAVE_MASKS == "flagged" and status == "FLAG"):
            for b in BONES:
                save_mask(raw[b], ct_img, MASK_DIR / vid / f"{b}.nii.gz")

        # --- QA visual for EVERY case ---
        title = status if status == "PASS" else f"{status}: " + "; ".join(hard + soft)
        qa_visual(vid, ct_arr, binar, title, FIG_DIR / f"{vid}.png")

        # --- export capped watertight STLs unless HARD-flagged (PASS or PASS_WARN export) ---
        if not hard:
            out_dir = GT_ROOT / case_id / side
            out_dir.mkdir(parents=True, exist_ok=True)
            for b in BONES:
                mesh = field_to_mesh(fields[b], ct_img)
                mesh.export(out_dir / f"{vid} segmentation_{b}.stl")
                if not (mesh.is_watertight and mesh.volume > 0):
                    print(f"    [warn] {vid} {b}: not watertight (is_watertight={mesh.is_watertight})")
        return row, diag_rows
    finally:
        shutil.rmtree(workdir, ignore_errors=True)


print("process_case loaded (persist masks + scored diagnostics + capped watertight export)")

## 4. Run

Processes every discovered case. A hard failure on one case is caught, logged as `ERROR`, and does
not stop the run. On CPU with `FAST=False` this is very slow — use `CASE_LIMIT=1` + `FAST=True` for
a local smoke test; run the full set on HPC GPU.

In [ ]:
import traceback

rows, diag_all = [], []
for i, w in enumerate(CASES, 1):
    print(f"[{i}/{len(CASES)}] {w['volume_id']} ...", flush=True)
    try:
        row, diags = process_case(w)
        diag_all.extend(diags)
    except Exception as exc:                       # keep going; record the failure
        traceback.print_exc()
        row = dict(volume_id=w["volume_id"], case_id=w["case_id"], side=w["side"],
                   status="ERROR", hard_flags=f"{type(exc).__name__}: {exc}")
    rows.append(row)
    print(f"    -> {row['status']}  {row.get('hard_flags','')}  {row.get('soft_flags','')}", flush=True)

manifest = pd.DataFrame(rows)
diagnostics = pd.DataFrame(diag_all)
manifest

## 5. Summary & reports

Write `manifest.csv` (all cases) and `flags.csv` (cases needing manual segmentation), and print the
PASS/flag breakdown.

In [ ]:
manifest.to_csv(REPORT_DIR / "manifest.csv", index=False)
diagnostics.to_csv(REPORT_DIR / "diagnostics.csv", index=False)
flagged = manifest[manifest["status"].isin(["FLAG", "ERROR"])]
flagged.to_csv(REPORT_DIR / "flags.csv", index=False)

n_pass = int((manifest["status"] == "PASS").sum())
n_warn = int((manifest["status"] == "PASS_WARN").sum())
print(f"cases processed   : {len(manifest)}")
print(f"PASS  (exported)  : {n_pass}")
print(f"PASS_WARN (export): {n_warn}   (soft issues, still exported -> review)")
print(f"FLAG/ERROR (skip) : {len(flagged)}   -> salvage in vsd_gt_watertight.ipynb or manual")
print(f"STL files written : {(n_pass + n_warn) * len(BONES)}")

# per-reason counts (a case can carry several reasons; strip the :bone suffix)
from collections import Counter
reasons = Counter()
for col in ("hard_flags", "soft_flags"):
    for f in manifest.get(col, pd.Series(dtype=str)).fillna(""):
        for r in str(f).split(";"):
            r = r.strip()
            if r:
                reasons[r.split(":")[0]] += 1
if reasons:
    print("\nreason counts (hard + soft):")
    for r, c in reasons.most_common():
        sev = "HARD" if r in HARD_FLAGS else ("SOFT" if r in SOFT_FLAGS else "?")
        print(f"  {r:16s} [{sev}] {c}")

print("\ncases routed to fallback / manual (HARD-flagged):")
for _, r in flagged.iterrows():
    print(f"  {r['volume_id']:18s} {r['status']:5s} {r.get('hard_flags','')}")

print(f"\nreports -> {REPORT_DIR}  (manifest.csv, diagnostics.csv, flags.csv)")
print(f"masks   -> {MASK_DIR}/<volume_id>/<bone>.nii.gz  (feeds vsd_gt_watertight.ipynb)")
print(f"QA PNGs -> {FIG_DIR}")
print(f"STLs    -> {GT_ROOT}/<CaseID>/<Side>/")

## 6. Frame round-trip check (self-verification)

Re-voxelize an exported STL back onto its source CT with the **same convention** the downstream
pipeline uses (`gt_per_bone.ipynb::voxelize_on`) and confirm the mesh lands on bone (high fraction
of re-voxelized voxels on HU-bright CT). This proves the STL is written in the correct raw-CT LPS
world frame. Runs on the first PASS case.

In [ ]:
def voxelize_on(stl_path, ref_img):
    """Solid-rasterize an STL (world-LPS mm) onto ref_img's grid — verbatim convention from
    notebooks/modeling/gt_per_bone.ipynb (the downstream consumer)."""
    mesh = trimesh.load(str(stl_path), process=False)
    v = np.asarray(mesh.vertices)
    idx = np.array([ref_img.TransformPhysicalPointToContinuousIndex(tuple(map(float, p))) for p in v])
    mesh.vertices = idx[:, ::-1]                            # (i,j,k)->(k,j,i)=(z,y,x)
    pts = np.round(np.asarray(mesh.voxelized(pitch=1.0).fill().points)).astype(int)
    shape = sitk.GetArrayFromImage(ref_img).shape          # (z,y,x)
    m = np.zeros(shape, np.uint8)
    ok = (pts >= 0).all(1) & (pts[:, 0] < shape[0]) & (pts[:, 1] < shape[1]) & (pts[:, 2] < shape[2])
    pts = pts[ok]; m[pts[:, 0], pts[:, 1], pts[:, 2]] = 1
    return m


_exported = manifest[manifest["status"].isin(["PASS", "PASS_WARN"])]
if len(_exported) == 0:
    print("no exported case to round-trip (check flags.csv)")
else:
    w = next(w for w in CASES if w["volume_id"] == _exported.iloc[0]["volume_id"])
    ct_img = sitk.ReadImage(str(ensure_nifti(w["input_path"], SCRATCH)))
    ct_arr = sitk.GetArrayFromImage(ct_img)
    stl = GT_ROOT / w["case_id"] / w["side"] / f"{w['volume_id']} segmentation_femur.stl"
    mesh = trimesh.load(str(stl), process=False)
    vox = voxelize_on(stl, ct_img)
    n = int(vox.sum())
    on_bone = float((ct_arr[vox > 0] > HU_BONE).mean()) if n else 0.0
    print(f"round-trip case : {w['volume_id']}  ({stl.name})")
    print(f"is_watertight   : {mesh.is_watertight}   volume(mm^3): {mesh.volume:.0f}")
    print(f"re-voxelized    : {n} voxels   fraction on bone (HU>{HU_BONE}): {on_bone:.3f}")
    assert mesh.is_watertight, "exported femur STL is NOT watertight — capping failed"
    assert n > MIN_VOXELS, "re-voxelized femur is empty — frame/convention mismatch"
    assert on_bone > 0.5, f"femur STL does not land on bone ({on_bone:.2f}) — frame mismatch"
    print("FRAME + WATERTIGHT ROUND-TRIP PASS: STL is watertight and in the correct raw-CT LPS frame")

## 7. Offline validation (no TotalSegmentator / GPU)

Synthetic-mask unit tests for the two risky pieces — **watertight capping** and the **diagnostic
logic** — so the notebook can be validated locally before the HPC run. Uses the notebook's own
helper functions on hand-built masks with a non-trivial CT geometry.

In [ ]:
# synthetic CT geometry (anisotropic spacing, non-zero origin) shared by the tests
_ct = sitk.GetImageFromArray(np.zeros((60, 70, 80), np.float32))
_ct.SetSpacing((0.86, 0.86, 0.6)); _ct.SetOrigin((-120.5, -188.0, 1275.0))
zz, yy, xx = np.ogrid[:60, :70, :80]

# --- Test 1: capping -> watertight, even when the bone is cut by the FOV (touches z0/z1) ---
cyl = (((yy - 35) ** 2 + (xx - 40) ** 2) <= 12 ** 2).astype(np.uint8)
cyl = np.broadcast_to(cyl, (60, 70, 80)).copy()          # full-height cylinder: touches z0 AND z1
assert touched_faces(cyl) >= {"z0", "z1"}, "test cylinder should touch both axial ends"
mesh = field_to_mesh(smooth_field(cyl, _ct), _ct)
print(f"[capping] touches={sorted(touched_faces(cyl))}  is_watertight={mesh.is_watertight}  "
      f"volume={mesh.volume:.0f}mm^3")
assert mesh.is_watertight, "capped cut cylinder must be watertight"
assert mesh.volume > 0, "capped mesh must enclose positive volume"

# --- Test 2: connectivity -> two separated blobs read as 2 components ---
two = np.zeros((60, 70, 80), np.uint8)
two[10:20, 10:25, 10:25] = 1; two[40:50, 45:60, 45:60] = 1
assert n_components(two) == 2, f"expected 2 components, got {n_components(two)}"

# --- Test 3: truncation -> a patella-like blob touching a lateral face is detected ---
pat = np.zeros((60, 70, 80), np.uint8); pat[25:35, 30:45, 0:12] = 1   # touches x0
assert "x0" in touched_faces(pat), "boundary contact not detected"

# --- Test 4: inter-bone overlap fraction ---
a = np.zeros((60, 70, 80), np.uint8); a[20:40, 20:40, 20:40] = 1
b = np.zeros((60, 70, 80), np.uint8); b[30:50, 20:40, 20:40] = 1      # ~50% overlap with a
ov = pairwise_overlap_frac(a, b)
assert ov > OVERLAP_FRAC, f"overlap {ov:.2f} should exceed OVERLAP_FRAC={OVERLAP_FRAC}"

# --- Test 5: hu_score low when mask sits on soft tissue (HU < HU_BONE) ---
soft_ct = np.full((60, 70, 80), 50.0, np.float32)        # 50 HU everywhere (soft tissue)
assert hu_score(a, soft_ct) < HU_MIN_FRAC, "hu_score should be low on soft-tissue CT"
soft_ct[a > 0] = 400.0                                    # now the mask sits on bone
assert hu_score(a, soft_ct) > HU_MIN_FRAC, "hu_score should be high when mask is on bone"

# --- Test 6: volume_mm3 matches voxel-count * spacing ---
vol = volume_mm3(a, _ct)
assert abs(vol - int(a.sum()) * 0.86 * 0.86 * 0.6) < 1e-3

print("OFFLINE VALIDATION PASS: capping watertight + all diagnostic checks behave correctly")